In [ ]:
import numpy as np
import torch
from torchvision import datasets
import math

In [ ]:
trainset = datasets.MNIST(root='./data', train=True, download=True)
testset = datasets.MNIST(root='./data', train=False, download=True)

In [ ]:
np.random.seed(0)
val_ratio = 0.1
train_size = len(trainset)
indices = list(range(train_size))
split_idx = int(np.floor(val_ratio * train_size))
np.random.shuffle(indices)
train_idx, val_idx = indices[split_idx:], indices[:split_idx]

In [ ]:
train_data = trainset.data[train_idx].float()/255
train_labels = trainset.targets[train_idx]
val_data = trainset.data[val_idx].float()/255
val_labels = trainset.targets[val_idx]
test_data = testset.data.float()/255
test_labels = testset.targets

train_data = train_data.reshape(54000, 784)
val_data = val_data.reshape(6000, 784)
test_data = test_data.reshape(10000, 784)

In [ ]:
class MaxPriorityQueue:
  def __init__(self, capacity=1000):
    self.capacity = capacity
    self.count = 0
    self.elements = [None] * capacity

  def parent(self, i): return (i - 1) // 2
  def left(self, i): return 2 * i + 1
  def right(self, i): return 2 * i + 2

  def _key(self, i):
    return i[0]

  def max_heapify(self, i):
    l, r = self.left(i), self.right(i)
    largest = i

    if l < self.count and self._key(self.elements[l]) > self._key(self.elements[largest]):
      largest = l
    if r < self.count and self._key(self.elements[r]) > self._key(self.elements[largest]):
      largest = r

    if largest != i:
      self.elements[i], self.elements[largest] = self.elements[largest], self.elements[i]
      self.max_heapify(largest)

  def insert(self, x):
    if self.count < self.capacity:
      i = self.count
      self.count += 1

      while i > 0 and self._key(self.elements[self.parent(i)]) < self._key(x):
        self.elements[i] = self.elements[self.parent(i)]
        i = self.parent(i)

      self.elements[i] = x

  def maximum(self):
    if self.count > 0:
      return self.elements[0]

  def extract_max(self):
    if self.count > 0:
      max_elem = self.elements[0]
      self.elements[0] = self.elements[self.count - 1]
      self.count -= 1
      self.max_heapify(0)
      return max_elem

  def __len__(self):
    return self.count

  def __str__(self):
    return str(self.elements[:self.count])


In [ ]:
def euclidian_dist(v1, v2):
  return math.sqrt(sum([(v1[i]-v2[i])**2 for i in range(len(v1))]))

In [ ]:
def knn(data, item, labels, k = 5):
  idxs = MaxPriorityQueue(k)
  for i in range(len(data)):
    dist = euclidian_dist(data[i], item)
    if idxs.count < 5:
      idxs.insert((dist, labels[i]))
    else:
      idxs.extract_max()
      idxs.insert((dist, labels[i]))
  neighbors = [idxs.extract_max()[1] for i in range(5)][::-1]
  return neighbors

In [ ]:
knn(train_data, test_data[0], train_labels)

[tensor(7), tensor(7), tensor(7), tensor(7), tensor(4)]

In [ ]:
def knn_broadcasting(data, item, labels, k=5):
  diff = data - item
  dist_arr = torch.sum(diff**2, dim=1)
  _, idxs = torch.topk(dist_arr, k, largest=False)
  return labels[idxs]

In [ ]:
knn_broadcasting(train_data, test_data[0], train_labels)

tensor([7, 7, 7, 7, 7])

In [ ]:
def knn_broadcasting_all(data, test, labels, k=5):
  diff = data[:, None, :] - test[None, :, :]
  dist_arr = torch.sum(diff**2, dim=1).T
  _, idxs = torch.topk(dist_arr, k, largest=False)
  return labels[idxs]

In [ ]:
knn_broadcasting_all(train_data, test_data, train_labels)

In [ ]:
def knn_broadcasting_all_optimized(data, test, labels, k=5):
  n, d = data.shape
  m, _ = test.shape

  data_norms = (data**2).sum(dim=1).view(n, 1)
  test_norms = (test**2).sum(dim=1).view(1, m)
  cross = test @ data.T
  dist_arr = test_norms.T + data_norms.T - 2 * cross
  _, idxs = torch.topk(dist_arr, k, largest=False)
  return labels[idxs]

In [ ]:
idxs = knn_broadcasting_all_optimized(train_data, test_data, train_labels, k = 20)

In [ ]:
predictions = torch.tensor([])
for i in range(len(idxs)):
  item = torch.mode(idxs[i]).values
  predictions = torch.cat([predictions, torch.tensor([item.item()])], dim=0)

predictions

tensor([7., 2., 1.,  ..., 4., 5., 6.])

In [ ]:
accuracy = (predictions == test_labels)
accuracy.float().mean().item()

0.9610999822616577